# 1. Quick Look at the Data Structure with Pandas

When starting a new machine learning project, the first step is to understand the dataset. Here are the essential Pandas functions we use:

*   `df.head()`: Displays the first 5 rows of the dataset. It helps to see what the features look like.
*   `df.info()`: Provides a concise summary of the DataFrame. It shows the total number of rows, column data types, and the count of non-null values (very useful for spotting missing data).
*   `df["column_name"].value_counts()`: Counts the occurrences of each unique category in a categorical (object) column.
*   `df.describe()`: Generates a statistical summary of numerical columns (e.g., mean, min, max, and percentiles).
*   `df.hist(bins=50)`: Plots a histogram to visualize the distribution of numerical data. The `bins` parameter defines the number of intervals the data is divided into.

# 2. Creating a Test Set

Before training a machine learning model, we must set aside a portion of our dataset (usually 20%) as the **Test Set**. 

**Why?**
1. To evaluate how well the model performs on unseen data (Generalization).
2. To avoid Data Leakage (Data Snooping Bias) during preprocessing steps like standardization.

```python
from sklearn.model_selection import train_test_split

# Splitting the data into 80% training and 20% testing
train_set, test_set = train_test_split(housing_full, test_size=0.2, random_state=42)

**The Importance of `random_state`**
Setting a random seed (e.g., `random_state=42`) ensures that the random split is identical every time the code runs. Without it, the test set would change on every run, and the model would eventually see the entire dataset, leading to data leakage. The number 42 is just a popular convention.

# 3. Stratified Sampling

Random sampling is fine for large datasets, but it can introduce sampling bias if the dataset is small or has important minority groups. 

**Stratified Sampling** ensures that the train and test sets have the exact same proportion of an important feature as the overall dataset. In this project, `median_income` is crucial for predicting housing prices. Therefore, we create income categories (`income_cat`) and stratify the split based on it.

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Create income categories
housing_full["income_cat"] = pd.cut(housing_full["median_income"],
                                    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                                    labels=[1, 2, 3, 4, 5])

# 2. Stratified split based on the income category
strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2, stratify=housing_full["income_cat"], random_state=42)

# 3. Drop the 'income_cat' column as we don't need it anymore
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

# 4. Discover and Visualize the Data

Visualizing geographical data is a great way to find patterns. By plotting longitude (x-axis) and latitude (y-axis), we essentially draw a map of California!

```python
# Create a copy of the training set to play with
housing = strat_train_set.copy()

# Plotting geographical data
# alpha=0.2 makes the points transparent, helping us see the density of data points
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.2)

### Adding More Dimensions: Price and Population

To visualize more than two features on a 2D scatter plot, we can use the size and color of the points:
*   `s` (size): Represents the population of the district. Larger circles mean higher population.
*   `c` (color): Represents the median house value. We use a predefined color map (`cmap="jet"`) where blue is cheap and red is expensive.

```python
import matplotlib.pyplot as plt

housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,
             s=housing["population"]/100, label="population", figsize=(10,7),
             c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True,
             sharex=False)
plt.legend()

# 5. Looking for Correlations

We can compute the **Standard Correlation Coefficient (Pearson's r)** to see how much each feature correlates with the median house value. The correlation coefficient ranges from -1 to 1:
*   **Close to 1:** Strong positive correlation (e.g., as income goes up, house value goes up).
*   **Close to -1:** Strong negative correlation (e.g., going further north might decrease prices).
*   **Close to 0:** No linear correlation.

```python
# Compute the correlation matrix
corr_matrix = housing.corr(numeric_only=True)

# Let's look at how much each feature correlates with the median house value
corr_matrix["median_house_value"].sort_values(ascending=False)

### Visualizing Correlations

Another way to check for correlations between attributes is to use the Pandas `scatter_matrix` function, which plots every numerical attribute against every other numerical attribute. 

Since the most promising attribute to predict the house value is the `median_income`, let's zoom in on their correlation scatter plot:

```python
from pandas.plotting import scatter_matrix

# We focus on a few promising attributes
attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12, 8))

# Zooming in on the strong relationship between median income and median house value
housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.1)

# 6. Experimenting with Attribute Combinations

Sometimes, the given features are not very useful on their own. For example, the total number of rooms in a district (`total_rooms`) isn't helpful if you don't know how many households there are. 

We can create new, more meaningful features by combining existing ones. For instance, the number of rooms per household:

```python
# Creating new combined features
housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"] / housing["households"]

# Check the correlation matrix again to see if the new features are useful
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

# 7. Prepare the Data for Machine Learning Algorithms

First, let's separate the predictors (features) and the labels (target variable). We don't want to apply the same transformations to the predictors and the target values.

```python
# drop() creates a copy of the data and does not affect strat_train_set
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

### Data Cleaning

Most Machine Learning algorithms cannot work with missing features. We have 3 options:
1. Get rid of the corresponding districts (rows).
2. Get rid of the whole attribute (column).
3. Set the values to some value (zero, the mean, the median, etc.) - called **Imputation**.

We will use Scikit-Learn's `SimpleImputer` to replace missing values with the median of that attribute:

```python
from sklearn.impute import SimpleImputer

# 1. Create an imputer instance, specifying that we want to replace missing values with the median
imputer = SimpleImputer(strategy="median")

# 2. Median can only be computed on numerical attributes, so we create a copy without the text attribute 'ocean_proximity'
housing_num = housing.drop("ocean_proximity", axis=1)

# 3. Fit the imputer to the training data (computes the median of each attribute)
imputer.fit(housing_num)

# 4. Use the trained imputer to transform the training set by replacing missing values
X = imputer.transform(housing_num)

# The result is a plain NumPy array, let's put it back into a Pandas DataFrame
import pandas as pd
housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)

# 8. Outlier Detection with Isolation Forest

Outliers are extreme values that deviate significantly from other observations. They can heavily skew the results of machine learning models (especially those sensitive to mean or variance). 

To automatically detect and remove outliers, we can use the **Isolation Forest** algorithm. It works by building a random forest; since outliers are few and different, they get isolated closer to the root of the trees (requiring fewer random splits).

```python
from sklearn.ensemble import IsolationForest

# 1. Create the Isolation Forest model
# random_state ensures reproducibility
isolation_forest = IsolationForest(random_state=42)

# 2. Fit the model and predict outliers
# It returns 1 for normal instances (inliers) and -1 for anomalies (outliers)
outlier_pred = isolation_forest.fit_predict(housing_num)

# 3. See the results
outlier_pred

# Optional: How to drop the outliers from the dataset
# housing = housing.iloc[outlier_pred == 1]
# housing_labels = housing_labels.iloc[outlier_pred == 1]

# 9. Handling Text and Categorical Attributes

Most machine learning algorithms prefer to work with numbers. Our dataset has one text attribute: `ocean_proximity`. 

### Why not simple Ordinal Encoding (0, 1, 2...)?
If we map categories to numbers like `0`, `1`, and `2`, machine learning algorithms might assume that category 2 is "greater" or "better" than category 0, which is incorrect for categories with no natural ordering.

### One-Hot Encoding
To fix this, we use **One-Hot Encoding**. This creates a binary column for each category (e.g., if there are 3 categories, we get 3 new columns filled with 0s and 1s). Only one of these columns will be 1 (hot) for any given sample.

```python
from sklearn.preprocessing import OneHotEncoder

# Create the encoder
cat_encoder = OneHotEncoder()

# Transform the categorical column into a one-hot matrix
housing_cat_1hot = cat_encoder.fit_transform(housing[["ocean_proximity"]])

# To see the categories it found:
cat_encoder.categories_

# 10. Feature Scaling

Machine learning algorithms generally don't perform well when the input numerical attributes have very different scales. In our housing data:
* `total_rooms` ranges from 6 to 39,320.
* `median_income` ranges from 0.4999 to 15.0001.

Without scaling, algorithms might mistakenly give much more weight to `total_rooms` simply because its numbers are larger. To fix this, we ensure all features have a similar scale. 

There are two common ways to get all attributes to have the same scale:
1. **Min-Max Scaling (Normalization):** Values are shifted and rescaled so they end up ranging from 0 to 1. (Using `MinMaxScaler`).
2. **Standardization:** Subtracts the mean value (so standardized values always have a zero mean), and then it divides by the standard deviation. Standardization is much less affected by outliers. (Using `StandardScaler`).

```python
from sklearn.preprocessing import StandardScaler

# 1. Create the StandardScaler instance
scaler = StandardScaler()

# 2. Fit the scaler to the numerical data and transform it simultaneously
# 'fit' calculates the mean and standard deviation for each feature.
# 'transform' subtracts the mean and divides by the standard deviation.
housing_num_scaled = scaler.fit_transform(housing_num)

# Now, all numerical features have a mean of ~0 and a variance of 1.
```



##### Let's take a look to MinMaxScaler in Scikit-Learn
```python
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
housing_num_min_max_scaled = min_max_scaler.fit_transform(housing_num)

# 11. Scaling the Target Variable (Labels)

In most cases, we only scale the input features (X). However, sometimes the target variable (y) has a heavy tail or very large values, which can make it difficult for algorithms like Linear Regression to find the underlying patterns. In such cases, we can scale or transform the target variable as well.

### Manual Approach vs. TransformedTargetRegressor

If we transform the target variable manually before training, we must remember to apply the **inverse transformation** to any predictions the model makes so we get the actual values (e.g., house prices in dollars) instead of scaled numbers.

To automate this, Scikit-Learn provides `TransformedTargetRegressor`. It wraps around your regression model and your chosen scaler. It automatically scales the labels during `fit()`, and automatically applies the inverse transformation during `predict()`.

```python
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor

# Wrap the Linear Regression model and the StandardScaler together
model = TransformedTargetRegressor(
    LinearRegression(),
    transformer=StandardScaler()
)

# We fit the model using the ORIGINAL unscaled labels. 
# The wrapper handles the scaling internally.
model.fit(housing[["median_income"]], housing_labels)

# Predictions are automatically inverse-transformed back to original scale (dollars)
predictions = model.predict(some_new_data)

# 12. Custom Transformers

Although Scikit-Learn provides many useful transformers, you will sometimes need to write your own for tasks such as applying a specific mathematical function, custom cleanup operations, or combining specific attributes.

### Using `FunctionTransformer`
For simple transformations that don't need to "learn" anything from the data (no `fit` method needed), you can use a `FunctionTransformer`. For example, features with a heavy-tailed distribution (like `population`) can be compressed into a more symmetrical, bell-shaped distribution by taking their logarithm.

### Log Transformation for Heavy-Tailed Distributions
Machine learning models perform best when numerical features have a symmetrical, bell-shaped distribution (Normal distribution). However, features like `population` or `median_income` often have a "heavy tail" (many districts have small populations, but a few have huge ones). 

Applying a logarithmic function (`np.log`) compresses these extreme large values. This makes the distribution much more symmetrical, helping algorithms discover patterns more effectively.

```python
from sklearn.preprocessing import FunctionTransformer
import numpy as np

# 1. Create a transformer that applies the logarithm
log_transformer = FunctionTransformer(np.log, inverse_func=np.exp)

# 2. Apply it to the heavily skewed 'population' column
log_pop = log_transformer.transform(housing[["population"]])

### Geographic Similarity (RBF Kernel)
Raw latitude and longitude are not always easy for a model to interpret on their own. However, house prices are heavily influenced by their distance to major economic hubs (like San Francisco).

We can create a new feature that measures the "geographic similarity" (or closeness) of each district to a specific landmark using a Radial Basis Function (**RBF kernel**). The RBF kernel outputs a value close to 1 if the district is very close to the landmark, and approaches 0 as it gets further away.

```python
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import FunctionTransformer

# 1. Define the exact coordinates of the landmark (San Francisco)
sf_coords = 37.7749, -122.41

# 2. Create a transformer using the rbf_kernel
# 'Y' is the reference point we are measuring distance to.
sf_transformer = FunctionTransformer(rbf_kernel, kw_args=dict(Y=[sf_coords], gamma=0.1))

# 3. Calculate the similarity score for each district based on its latitude and longitude
sf_simil = sf_transformer.transform(housing[["latitude", "longitude"]])

### Custom Transformers with Classes

For transformations that need to "learn" parameters from the data (like means, standard deviations, or cluster centers) before applying them, we must create a custom Python class. This class inherits from `BaseEstimator` and `TransformerMixin`, and implements two main methods: `fit()` and `transform()`.

#### 1. Example: A Custom Standard Scaler
This acts exactly like Scikit-Learn's `StandardScaler`:
*   `fit()`: Looks at the data to learn the mean and standard deviation.
*   `transform()`: Applies the mathematical scaling using those learned parameters.

```python
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted

class StandardScalerClone(BaseEstimator, TransformerMixin):
    def __init__(self, with_mean=True):  # no *args or **kwargs!
        self.with_mean = with_mean

    def fit(self, X, y=None):  # y is required even though we don't use it
        X = check_array(X)  # checks that X is an array with finite float values
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.n_features_in_ = X.shape[1]  # every estimator stores this in fit()
        return self  # always return self!

    def transform(self, X):
        check_is_fitted(self)  # looks for learned attributes (with trailing _)
        X = check_array(X)
        assert self.n_features_in_ == X.shape[1]
        if self.with_mean:
            X = X - self.mean_
        return X / self.scale_
```

#### 2. Advanced Example: Cluster Similarity
Instead of manually defining one specific city (like San Francisco), we can use a clustering algorithm (`KMeans`) to automatically find multiple geographical hubs in our housing data. Then, we can measure the similarity of each district to all of these hubs.

```python
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  # always return self!

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]
```

# 13. Transformation Pipelines

As you can see, there are many data transformation steps that need to be executed in the right order. Scikit-Learn provides the `Pipeline` class to help with sequences of transformations. 

Think of a Pipeline as an assembly line: data goes in one end, passes through a series of transformers sequentially, and comes out fully processed at the other end.

Here is a small pipeline for the numerical attributes. We use `make_pipeline` which is a convenient shortcut that automatically names the steps for us:

```python
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Build the pipeline
num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

# 2. Fit and transform the numerical data in one go
housing_num_prepared = num_pipeline.fit_transform(housing_num)

# 3. Convert the resulting NumPy array back into a Pandas DataFrame
df_housing_num_prepared = pd.DataFrame(
    housing_num_prepared, 
    columns=num_pipeline.get_feature_names_out(),
    index=housing_num.index
)

df_housing_num_prepared.head(2)

### Inspecting and Modifying Pipelines

Once a pipeline is built, you might want to inspect a specific step or change its parameters without rebuilding the whole pipeline from scratch.

You can access individual steps using their index (e.g., `num_pipeline[1]`) or by their name using the `named_steps` dictionary. You can also update parameters dynamically using the `set_params()` method.

```python
# Accessing the SimpleImputer step by its automatically generated name
imputer_step = num_pipeline.named_steps["simpleimputer"]

# Changing a parameter inside the pipeline (e.g., changing strategy to "mean")
# The syntax is: step_name + two underscores + parameter_name
num_pipeline.set_params(simpleimputer__strategy="median")

# 14. Combining Pipelines with ColumnTransformer

We now have different pipelines for different types of data: one for numerical attributes and another for categorical attributes. To apply them all to the same dataset simultaneously, Scikit-Learn provides the `ColumnTransformer`.

It acts as a dispatcher: it routes specific columns to specific transformers or pipelines, and then concatenates the final results. We can use `make_column_selector` to automatically select columns based on their data type (e.g., all numbers or all objects/text).

# 15. The Ultimate Preprocessing Pipeline

Now we can bring everything together. We will build a single, massive `ColumnTransformer` that applies:
1. Custom ratios (e.g., bedrooms per room).
2. Log transformations (to fix heavy-tailed distributions like population).
3. Geographic similarity (distance to cluster centers).
4. Categorical encoding (One-Hot Encoding for text).
5. A default scaling for any remaining columns.

```python
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
import numpy as np

# Define custom functions
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler()
    )

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler()
)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

# The Mega-Transformer!
preprocessing = ColumnTransformer([
        ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline(), ["population", "households"]),
        ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                               "households", "median_income"]),
        ("geo", cluster_simil, ["latitude", "longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder=default_num_pipeline) # Handle anything else

# Run the entire raw dataset through the pipeline
housing_prepared = preprocessing.fit_transform(housing)

### Understanding the Output: Why 24 Columns?

When we pass our original dataset (which had 9 features) through the `ColumnTransformer`, the original DataFrame remains untouched. Instead, the transformer builds a brand new NumPy array (`housing_prepared`) by concatenating the outputs of all the pipelines. 

Depending on the transformer, the number of columns changes:
1.  **Ratios Pipeline (Many-to-One):** Took multiple columns and created **3 new columns** (e.g., bedrooms per room).
2.  **Log Pipeline (One-to-One):** Applied logarithm and scaling to 5 features, outputting **5 columns** with transformed values.
3.  **Cluster Similarity (One-to-Many):** Took latitude and longitude and calculated the distance to 10 cluster centers, outputting **10 new columns**.
4.  **Categorical Pipeline (One-to-Many):** `OneHotEncoder` turned 1 text column into **5 binary columns**.
5.  **Remainder (One-to-One):** The `housing_median_age` column was passed through the default scaler, outputting **1 column**.

Total columns in the new array: 3 + 5 + 10 + 5 + 1 = **24 columns**. This feature engineering process gives our machine learning model much richer information to learn from!

# Select and Train a Model

## 1. Linear Regression
At last, we have framed the problem, obtained and explored the data, sampled a training set and a test set, and written transformation pipelines to clean up and prepare the data for Machine Learning algorithms automatically.

Now we are ready to select and train a Machine Learning model. We will start with a simple Linear Regression model. Notice how we can append the model directly to the end of our preprocessing pipeline!

```python
from sklearn.linear_model import LinearRegression

# Combine preprocessing and the model into a single pipeline
lin_reg = make_pipeline(preprocessing, LinearRegression())

# Train the model using the RAW training data
lin_reg.fit(housing, housing_labels)

### Why do we pass `housing` instead of `housing_prepared`?

You might wonder why we call `lin_reg.fit(housing, housing_labels)` using the raw dataset instead of the 24-column `housing_prepared` array we just created. 

The secret lies in the `make_pipeline` function. Our new `lin_reg` pipeline consists of two steps:
1.  **`preprocessing`**: The massive `ColumnTransformer` we built.
2.  **`LinearRegression()`**: The machine learning model.

When we pass the raw `housing` data into this pipeline, it first goes through the `preprocessing` step. This step cleans the data, engineers the new features, and outputs the 24-column array internally. This array is then **automatically** passed to the `LinearRegression` model for training. 

If we passed `housing_prepared` into this pipeline, the `preprocessing` step would crash because it expects raw data with specific column names (like "ocean_proximity" or "total_rooms"), not an already processed array!

## 2. Decision Tree Regressor

Since the Linear Regression model underfitted the data, we will try a more complex algorithm: a `DecisionTreeRegressor`. 

A Decision Tree works by splitting the data into smaller and smaller subsets based on feature values (like playing a game of 20 Questions) until it groups similar houses together. 

```python
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

# 1. Create a new pipeline with the Decision Tree
tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))

# 2. Train the model
tree_reg.fit(housing, housing_labels)

# 3. Predict and evaluate on the training set
housing_predictions = tree_reg.predict(housing)
tree_rmse = root_mean_squared_error(housing_labels, housing_predictions)
print(f"Decision Tree RMSE: {tree_rmse}")

### The Problem of Overfitting
##### The code above outputs an RMSE of exactly 0.0!

Does this mean we have found an absolutely perfect model? No. It is highly likely that the model has badly overfit the data. A Decision Tree, if not constrained, will keep splitting the data until it has memorized the exact price of every single house in the training set. It knows the training data perfectly, but it will likely perform terribly on new, unseen data.

## Better Evaluation Using Cross-Validation
To truly test a model without touching the Test Set, we use **K-fold Cross-Validation**. It splits the training data into multiple parts and tests the model on unseen folds.

```python
from sklearn.model_selection import cross_val_score
import pandas as pd

tree_rmses = -cross_val_score(tree_reg, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10)
pd.Series(tree_rmses).describe()
```
*Note: Scikit-Learn’s cross-validation features expect a utility function (greater is better) rather than a cost function (lower is better), so the scoring function is the opposite of the MSE (a negative value), which is why we add a minus sign before calculating the square root.*

*(Cross-validation revealed the Decision Tree's true error is ~$67,013).*

## 3. Random Forest Regressor
Random Forests train many Decision Trees on random subsets and average their predictions (**Ensemble Learning**).

```python
from sklearn.ensemble import RandomForestRegressor
forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=42))

forest_rmses = -cross_val_score(forest_reg, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10)
pd.Series(forest_rmses).describe()
```

# Fine-Tune Your Model

Once we have a shortlist of promising models, we need to fine-tune them. We can use Scikit-Learn's `GridSearchCV` to automatically experiment with different combinations of hyperparameters for both our data preparation steps and our machine learning model.

## Grid Search

```python
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Step 1: Re-create the full pipeline
full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])

# Step 2: Define the hyperparameter grid to test
param_grid = [
    # First group: 3 cluster options * 3 feature options = 9 combinations
    {'preprocessing__geo__n_clusters': [5, 8, 10],
     'random_forest__max_features': [4, 6, 8]},
    # Second group: 2 cluster options * 3 feature options = 6 combinations
    {'preprocessing__geo__n_clusters': [10, 15],
     'random_forest__max_features': [6, 8, 10]},
]
# Total of 15 combinations will be tested

# Step 3: Set up the grid search
# cv=3 means 3-fold cross-validation for each combination (15 * 3 = 45 training rounds)
grid_search = GridSearchCV(full_pipeline, param_grid, cv=3,
                           scoring='neg_root_mean_squared_error')

# Step 4: Run the search on the raw training data
grid_search.fit(housing, housing_labels)
```

#### The Best Parameters
After running for a while, it finds the optimal combination

```python
# Display the best hyperparameters found
grid_search.best_params_
# Output: {'preprocessing__geo__n_clusters': 15, 'random_forest__max_features': 6}
```

## Randomized Search

When the hyperparameter search space is large, `GridSearchCV` becomes too slow because it evaluates every single combination. Instead, we can use `RandomizedSearchCV`. 

You give it a range (or a statistical distribution) for each hyperparameter, and it evaluates a specific number of random combinations by setting the `n_iter` parameter. This gives you much more control over the computing budget.

```python
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 1. Define the search space using probability distributions
# randint(low=3, high=50) means every integer between 3 and 49 is equally likely to be chosen.
param_distribs = {
    'preprocessing__geo__n_clusters': randint(low=3, high=50),
    'random_forest__max_features': randint(low=2, high=20)
}

# 2. Set up the Randomized Search
# n_iter=10 means it will only pick 10 random combinations to test.
# cv=3 means 3-fold cross-validation for each combination.
rnd_search = RandomizedSearchCV(
    full_pipeline, param_distributions=param_distribs, n_iter=10, cv=3,
    scoring='neg_root_mean_squared_error', random_state=42
)

# 3. Run the search
rnd_search.fit(housing, housing_labels)
```

### Analyzing the Randomized Search Results
Let's look at the scores of the 10 random combinations it tested:

```python
import pandas as pd
import numpy as np

# Display the random search results in a clean DataFrame
cv_res = pd.DataFrame(rnd_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)

# Select specific columns to show
score_cols = ["split0_test_score", "split1_test_score", "split2_test_score", "mean_test_score"]
cv_res = cv_res[["param_preprocessing__geo__n_clusters", 
                 "param_random_forest__max_features"] + score_cols]
cv_res.columns = ["n_clusters", "max_features"] + score_cols

# Convert negative RMSE to positive integers for readability
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)
cv_res.head()
```

### Bonus: Hyperparameter Sampling Distributions
To choose random values, `RandomizedSearchCV` relies on distributions from `scipy.stats`:
*   `randint(a, b+1)`: For discrete variables (integers) where all values are equally likely.
*   `uniform(a, b)`: For continuous variables where all values are equally likely.
*   `geom` and `expon`: When you want to sample roughly in a given scale, favoring smaller values.
*   `loguniform`: When you have no idea what the optimal scale is (e.g., 0.01 is as likely as 100).

```python
# Code to plot the probability distributions (Optional visual reference)
from scipy.stats import randint, uniform, geom, expon
import matplotlib.pyplot as plt

xs1 = np.arange(0, 7 + 1)
randint_distrib = randint(0, 7 + 1).pmf(xs1)

plt.figure(figsize=(12, 7))
plt.subplot(2, 2, 1)
plt.bar(xs1, randint_distrib, label="scipy.randint(0, 7 + 1)")
plt.ylabel("Probability")
plt.legend()
plt.axis([-1, 8, 0, 0.2])
plt.show()
```

## Analyze the Best Models and Their Errors

Before testing on the final test set, it's helpful to inspect the best model. For a `RandomForestRegressor`, we can look at the feature importances to see which features contribute the most to accurate predictions.

```python
# 1. Get the best model pipeline found by Randomized Search
final_model = rnd_search.best_estimator_

# 2. Extract feature importances from the Random Forest step
feature_importances = final_model["random_forest"].feature_importances_

# 3. Pair feature names with their importance scores and sort them in descending order
sorted(zip(feature_importances, final_model["preprocessing"].get_feature_names_out()), reverse=True)
```

---

## Evaluate Your System on the Test Set

Now that the model is fine-tuned, it's time to evaluate it on the untouched Test Set. This is the final exam for our machine learning model.

```python
from sklearn.metrics import root_mean_squared_error
from scipy import stats
import numpy as np

# 1. Separate the features and labels from the Test Set
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

# 2. Make predictions on the Test Set
# IMPORTANT: We use predict(), NEVER use fit() or fit_transform() on the test set!
final_predictions = final_model.predict(X_test)

# 3. Calculate the final RMSE
final_rmse = root_mean_squared_error(y_test, final_predictions)
print(f"Final RMSE: {final_rmse}")

# 4. Compute a 95% confidence interval for the test RMSE
# This tells us the range where the true error likely falls
confidence = 0.95
squared_errors = (final_predictions - y_test) ** 2
boot_result = stats.bootstrap([squared_errors], np.mean, confidence_level=confidence, random_state=42)
rmse_lower = np.sqrt(boot_result.confidence_interval.low)
rmse_upper = np.sqrt(boot_result.confidence_interval.high)

print(f"95% CI for RMSE: ({rmse_lower:.4f}, {rmse_upper:.4f})")
```





## Model Persistence using joblib

Once the project is complete, you should save your final model so you can move it to production, share it with others, or load it later without having to retrain it.

```python
import joblib

# 1. Save the final pipeline (preprocessing + model) to a file
joblib.dump(final_model, "my_california_housing_model.pkl")

# ---------------------------------------------------------
# 2. Simulating a Production Environment:
# Later, or in a completely different Python script, you can load the model
final_model_reloaded = joblib.load("my_california_housing_model.pkl")

# 3. Pretend 'new_data' contains raw data from new districts
new_data = housing.iloc[:5] 

# 4. The reloaded pipeline automatically cleans the data and makes predictions!
predictions = final_model_reloaded.predict(new_data)
print(predictions)
```

# 🌍 The Big Picture: End-to-End Machine Learning Workflow

This section is a high-level summary of everything we did in this project. In Machine Learning, you don't need to memorize the exact code, but you **must** understand this workflow.

### 1. Look at the Big Picture & Get the Data
*   **Goal:** Predict median house values in Californian districts.
*   **Action:** Downloaded the data and used Pandas (`head()`, `info()`, `describe()`, `hist()`) to get a feel for it. We noticed missing values and different scales.

### 2. Create a Test Set (Hide the Final Exam!)
*   **Concept:** Before looking too closely at the data, we must set aside 20% of it as a Test Set to avoid "Data Leakage" (cheating).
*   **Action:** Used `StratifiedShuffleSplit` based on the `income_cat` to ensure our test set accurately represents the whole population's income distribution.

### 3. Explore and Visualize the Data
*   **Action:** Created scatter plots (latitude vs. longitude) to see housing prices geographically.
*   **Discovery:** Found that median income is the strongest predictor of house prices. 

### 4. Prepare the Data for Machine Learning (The Conveyor Belt)
*   *This is usually the hardest and most code-heavy part! Algorithms need clean numbers.*
*   **Numerical Features:** Filled missing values using the median (`SimpleImputer`), created new logical features (like rooms per house), and scaled everything so algorithms treat them equally (`StandardScaler`).
*   **Categorical Features:** Converted text data (like `<1H OCEAN`) into numbers using `OneHotEncoder`.
*   **The Pipeline:** We combined all these steps into a single `ColumnTransformer` (our automated cleaning machine).

### 5. Select and Train a Model
*   **Action:** We trained a few simple models first. 
    *   `LinearRegression`: Underfitted the data (too simple).
    *   `DecisionTree`: Overfitted the data (memorized the training set).
    *   `RandomForest`: Worked best, showing a good balance.
*   **Validation:** Used Cross-Validation (`cv=10`) to test the models fairly without touching the final Test Set.

### 6. Fine-Tune the Model
*   **Concept:** Now that we have a good model, let's adjust its "dials" (Hyperparameters) to make it even better.
*   **Action:** Used `RandomizedSearchCV` to automatically test different random combinations of parameters and find the best one without waiting days for the computer to finish.

### 7. Evaluate on the Test Set
*   **Action:** Put the final, fine-tuned model to the test on the untouched 20% of data. Calculated the final RMSE and the 95% Confidence Interval to know the real-world error range.

### 8. Save the Model
*   **Action:** Used `joblib` to save the final pipeline. Now we can deploy it to production and feed it new data without rewriting any code.